In [9]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt 


In [ ]:
base = Path.cwd()
if not (base / "data" / "raw" / "meal_info.csv").exists():
    base = base.parent

meals = pd.read_csv(base / "data" / "raw" / "meal_info.csv")
centers = pd.read_csv(base / "data" / "raw" / "fulfilment_center_info.csv")
train = pd.read_csv(base / "data" / "raw" / "train.csv")


In [20]:
merged_data = train.merge(meals, on='meal_id').merge(centers, on='center_id')

In [27]:
# Split rows by whether the order count is negative.
negative_order_rows = merged_data[merged_data["num_orders"] < 0].copy()
nonnegative_order_rows = merged_data[merged_data["num_orders"] >= 0].copy()

print("Negative order-count rows:", len(negative_order_rows))
print("Non-negative order-count rows:", len(nonnegative_order_rows))

if not negative_order_rows.empty:
    print("\nRows to review:")
    print(negative_order_rows[["center_id", "meal_id", "week", "num_orders"]].head(10).to_string(index=False))
    print("\nRecommendation: verify these values against the source records. Since the true count cannot be inferred, mark negative counts as missing in a separate cleaned copy until they can be recovered.")
else:
    print("No negative order counts were found; no repair is needed.")

# Keep the merged source unchanged; mark any invalid negative counts as missing in a copy.
cleaned_data = merged_data.copy()
cleaned_data["num_orders"] = cleaned_data["num_orders"].mask(cleaned_data["num_orders"] < 0)
print("Negative values remaining in cleaned_data:", int((cleaned_data["num_orders"] < 0).sum()))

Negative order-count rows: 0
Non-negative order-count rows: 456548
No negative order counts were found; no repair is needed.
Negative values remaining in cleaned_data: 0


In [24]:
merged_data = train.merge(meals, on='meal_id').merge(centers, on='center_id')
print("the merged data (rows, columns)", merged_data.shape)
print("the total number of orders in the merged data:", merged_data['num_orders'].sum())

the merged data (rows, columns) (456548, 15)
the total number of orders in the merged data: 119557485


# check the data problems

In [32]:
print("Merged data shape:", merged_data.shape)

print("\nMissing values by column:")
missing_counts = merged_data.isna().sum()
missing_counts = missing_counts[missing_counts > 0]
print(missing_counts.to_string() if not missing_counts.empty else "No missing values")
if missing_counts.empty:
    print("Recommendation: no missing-value recovery is currently needed.")
else:
    print("Recommendation: inspect affected records and determine whether values are unavailable or not applicable; only then choose an evidence-based imputation or exclusion method.")

print("\nDuplicate checks:")
full_row_duplicates = merged_data.duplicated().sum()
key_duplicates = merged_data.duplicated(subset=["center_id", "meal_id", "week"]).sum()
print("Exact duplicate rows:", full_row_duplicates)
print("Duplicate center-meal-week keys:", key_duplicates)
if full_row_duplicates or key_duplicates:
    print("Recommendation: inspect duplicate records and compare with the source tables; remove records only if they are confirmed accidental duplicates.")
else:
    print("Recommendation: no duplicate recovery is currently needed.")

price_flags = merged_data[merged_data["checkout_price"] > merged_data["base_price"]]
print("\nRows where checkout_price > base_price:", len(price_flags))
if not price_flags.empty:
    print("Recommendation: verify these rows against the source and applicable pricing/promotion rules. Correct only confirmed data-entry errors; a higher checkout price may be legitimate.")
else:
    print("Recommendation: no price rows were flagged.")

order_q1, order_q3 = merged_data["num_orders"].quantile([0.25, 0.75])
order_iqr = order_q3 - order_q1
order_lower = order_q1 - 1.5 * order_iqr
order_upper = order_q3 + 1.5 * order_iqr
order_flags = merged_data[
    (merged_data["num_orders"] < order_lower)
    | (merged_data["num_orders"] > order_upper)
]
print(f"\nIQR order-count flags: {len(order_flags)} rows (below {order_lower:.2f} or above {order_upper:.2f})")
if not order_flags.empty:
    print(order_flags[["center_id", "meal_id", "week", "num_orders"]].sort_values("num_orders").head(5).to_string(index=False))
    print(order_flags[["center_id", "meal_id", "week", "num_orders"]].sort_values("num_orders", ascending=False).head(5).to_string(index=False))
    print("Recommendation: review flagged orders against neighboring weeks, promotions, and operational records. Treat IQR as a review flag, not proof of an error; retain valid demand spikes and correct only verified mistakes.")
else:
    print("Recommendation: no order-count outliers were flagged by the IQR rule.")

expected_weeks = set(range(1, 146))
missing_week_rows = []
for (center_id, meal_id), group in merged_data.groupby(["center_id", "meal_id"]):
    observed_weeks = set(group["week"].dropna().astype(int))
    gaps = sorted(expected_weeks - observed_weeks)
    if gaps:
        missing_week_rows.append({
            "center_id": center_id,
            "meal_id": meal_id,
            "missing_week_count": len(gaps),
            "first_missing_weeks": gaps[:10],
        })

missing_week_report = pd.DataFrame(missing_week_rows)
print("\nCenter-meal pairs with missing weeks:", len(missing_week_report))
if not missing_week_report.empty:
    print(missing_week_report.sort_values("missing_week_count", ascending=False).head(10).to_string(index=False))
    print("Recommendation: confirm whether each gap means the meal was not offered or records are missing. If unavailable, recover from the source system where possible; do not automatically fill with zero, since that would imply confirmed zero demand.")
else:
    print("Recommendation: no missing weeks were detected within weeks 1-145.")

Merged data shape: (456548, 15)

Missing values by column:
No missing values
Recommendation: no missing-value recovery is currently needed.

Duplicate checks:
Exact duplicate rows: 0
Duplicate center-meal-week keys: 0
Recommendation: no duplicate recovery is currently needed.

Rows where checkout_price > base_price: 116101
Recommendation: verify these rows against the source and applicable pricing/promotion rules. Correct only confirmed data-entry errors; a higher checkout price may be legitimate.

IQR order-count flags: 32937 rows (below -351.00 or above 729.00)
 center_id  meal_id  week  num_orders
       126     1778    60         730
       137     1062   136         730
        67     2290    18         730
        89     1778   132         730
        72     2707     9         730
 center_id  meal_id  week  num_orders
        43     2290     5       24299
        43     2290     7       15336
        43     2290     6       14229
        10     2290     5       13580
        43  

In [33]:
# Save the cleaned merged data as a new CSV; the original source files remain unchanged.
cleaned_merged_data = cleaned_data.copy()
output_path = base / "data" / "raw" / "cleaned_merged_data.csv"
cleaned_merged_data.to_csv(output_path, index=False)
print(f"Saved {cleaned_merged_data.shape[0]} rows to: {output_path}")

Saved 456548 rows to: c:\Users\HP\Desktop\food-demand-analysis\data\raw\cleaned_merged_data.csv
